# NB1 · Defining the problem

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---

Code generation is minimal in this notebook. The purpose is to define the system you
are going to build through seven questions, and to turn that definition into the input
for every prompt that follows.

Every prompt in the rest of the workshop carries this card. The vaguer the card, the
vaguer the code produced from it.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
               'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'en'

print('Ready.')


---

## Step 1 · Complete the card

The cell below holds the seven questions. It is filled in with the answers for the
shared scenario; replace that text with answers for your own problem.

Where a question cannot be answered, state what is unknown and why rather than leaving
it blank. A prevalence recorded as unknown is more useful than one left out entirely.
The first is passed to the prompt; the second silently becomes an assumption.


In [ ]:
CARD = {
    'decision': (
        'Predicting whether a patient admitted to intensive care will stay longer '
        'than three days. The system produces output six hours after admission and '
        'feeds bed capacity planning.'
    ),
    'user': (
        'The intensive care consultant and the nurse responsible for bed '
        'management, during the morning round as the patient list is reviewed.'
    ),
    'data_type': (
        'Routine hospital data. Demographics, admission context, and vital signs '
        'and laboratory results from the first six hours.'
    ),
    'outcome': (
        'Intensive care length of stay exceeding three days. It is computed from '
        'admission and discharge times and is present in routine records.'
    ),
    'prevalence': (
        'About one third in this cohort. The rate in your own unit will differ; if '
        'you do not know it, record that too.'
    ),
    'cost': (
        'A miss is more costly. Failing to identify a long stay fills capacity '
        'without a plan and delays the admission of the next patient. The cost of a '
        'false alarm is an unnecessary planning meeting.'
    ),
    'harm': (
        'On a false negative the capacity plan falls short and the transfer '
        'decision is delayed. Escalation rule: where the system abstains, or where '
        'the patient is unlike the training population, the decision passes to the '
        'responsible clinician.'
    ),
}


### Check 1


In [ ]:
checks.check_canvas(CARD)


---

## Step 2 · Turn the card into a specification

The card is written in ordinary language. To be usable in the notebooks that follow it
has to be given structure.

This step asks a generative AI tool to produce a specification from the card. The most
important part of the prompt is its final paragraph, which tells the tool not to fill in
gaps of its own accord. Without that instruction the tool produces a specification that
looks complete in every case and does not report what it invented.


### Prompt 1

Run the cell below, then copy the printed text into your AI tool.


In [ ]:
PROMPT = '''I am designing a clinical decision support system. My problem is:

{card}

Produce a specification from this definition and return it as a Python dictionary.

CONTRACT
Produce a Python dictionary named spec with exactly these keys:
  decision_point   -> the moment the system produces output, one sentence
  outcome_definition -> the precise definition of the outcome and its time window
  available_inputs -> list of variables accessible at the decision point
  forbidden_inputs -> list of variables accessible only afterwards
  priority         -> sensitivity or specificity, whichever should be favoured
  priority_reason  -> the reasoning for that choice, one paragraph
  harm_scenarios   -> three concrete ways the system could harm a patient, a list
  out_of_scope     -> what the system explicitly does not do, a list
  open_questions   -> gaps or contradictions in my answers, a list

Write the code so that it only builds the dictionary. Produce no other output.
Do not fill in anything you find missing or internally inconsistent; place those
points in open_questions. That list must not be empty.'''

print(PROMPT.format(card='\n\n'.join(f'{k}: {v}' for k, v in CARD.items())))


In [ ]:
# Paste the generated code into this cell and run it.


### Check 2


In [ ]:
checks.check_canvas(spec, min_chars=10)


---

## Reading the specification

Read the `open_questions` list first. It shows where your card is weak, and the same
points will be weak in the system you build from it. If the list comes back empty, be
suspicious; a definition of seven sentences rarely leaves no gaps.

Then compare `available_inputs` against `forbidden_inputs`. If the separation has
genuinely been made, the first defence against leakage is in place. If it has not,
correct the prompt and regenerate.

Finally check whether `priority` matches the cost balance you stated on the card. The
tool sometimes drifts to its own default and favours specificity in a problem where a
miss is the costly error.


In [ ]:
for key in ['priority', 'priority_reason']:
    print(f'{key}: {spec.get(key)}')
print()
print('Open questions:')
for question in spec.get('open_questions', []):
    print(' -', question)


## What this notebook covered

The problem was defined through seven questions and that definition was turned into a
structured specification for use in the notebooks that follow.

The card is the first and most decisive input given to the generative AI tool. Anything
absent from it is absent from every later prompt; the tool fills the gap by guessing and
does not report the guess.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
